# Problem Set 2: Runtime and Data Scaling

This workbook will help you benchmark your implementations of `countNucleotides`, `patternIndex`, `patternCount`, and `frequentWords`. First, make sure you complete the assignment and run `pset2-testing.ipynb`. The exercises in this workbook measure performance, not correctness.

Download this notebook and `pset2_benchmark.py` into the `pset-2` folder. The setup cell loads local `pset-2/src` and the sibling `pset-1/src` directly, so package installation is optional when both assignment folders are present. If `pset-1` is stored elsewhere, install both packages into the notebook environment as described in README.md. To generate plots, you may need to install matplotlib into this notebook's kernel. To do this: `%pip install matplotlib`.

The run writes two submission files: a CSV of individual measurements and a JSON with configuration, Python/platform information, timer precision, and hashes of your solution files. Use a class alias for your participant ID; no source code or hostname is included. Submit **both files from the run you want compared**. Each rerun creates new files.

In [1]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), Path.cwd() / "pset-2"]
             if (p / "pset2_benchmark.py").is_file() and (p / "src" / "pset2").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from pset-2 or its parent directory.")
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))

# The assignment imports helpers from the sibling pset-1 package.
PRIOR_SRC = ROOT.parent / "pset-1" / "src"
if (PRIOR_SRC / "frequent_words" / "__init__.py").is_file():
    sys.path.insert(0, str(PRIOR_SRC))

try:
    from pset2 import countNucleotides, patternIndex, patternCount, frequentWords
except ModuleNotFoundError as exc:
    if exc.name == "frequent_words":
        raise ModuleNotFoundError(
            "pset2 needs the prior assignment's frequent_words package. "
            "Place pset-1 beside pset-2, or install it in this notebook's kernel "
            "with %pip install -e ../pset-1, then restart the kernel."
        ) from exc
    raise
from pset2_benchmark import run_benchmarks, summarize, write_summary, load_results, plot_results

FUNCTIONS = {
    "countNucleotides": countNucleotides,
    "patternIndex": patternIndex,
    "patternCount": patternCount,
    "frequentWords": frequentWords,
}


## Choose your ID and run settings

Use the instructor's settings unchanged for comparable inputs. `n` is DNA length; `k` is fixed so this experiment varies only `n`. Pattern methods search for `"A" * k`. Random DNA uses a deterministic seed, and repeated DNA (`"A" * n`) produces many overlapping matches. These are two workloads, not all possible inputs or a proof of asymptotic complexity.

Start with the default sizes. Larger sizes may be useful for fast implementations; a quadratic `frequentWords` implementation can become slow quickly. The soft cutoff skips remaining samples and larger sizes for a method/workload **after** a slow call completes. It cannot interrupt that call; use Jupyter's Interrupt if necessary. Completed CSV rows remain saved and the JSON marks an interrupted run as incomplete. Exceptions (including unfinished stubs) are recorded and other methods continue.


In [2]:
PARTICIPANT_ID = "student-alias"  # Replace with your assigned ID or class alias.
SIZES = [100, 300, 1_000, 3_000, 10_000]
REPEATS = 3
K = 3
SEED = 258
SLOW_SECONDS = 1.0
OUTPUT_DIR = ROOT / "benchmark-results"


## Collect measurements

Each sample times one function call with `perf_counter`. Input generation, file writing, and return-size measurement are outside the timed interval. All samples, including the first call, are retained; medians reduce sensitivity to noise. Close busy applications and use the same kernel/environment throughout. If you edit your implementations, restart the kernel and rerun so imports pick up the changes.

Returned size includes (1) shallow bytes from `sys.getsizeof`, (2) deep bytes counting nested built-in objects once per identity, and (3) the number of returned items. An integer has one returned item; its numerical value is **not** a container length. Deep bytes estimate the storage reachable from the result, including shared Python objects; they do **not** measure extra allocated memory, peak memory during computation, or input size. Byte counts vary with Python version and architecture. A list of indices can grow with `n`; a four-key nucleotide dictionary or a scalar count generally stays small. The number of tied frequent words depends on the workload and `k`.


In [3]:
rows, csv_path, metadata_path = run_benchmarks(
    FUNCTIONS, participant_id=PARTICIPANT_ID, sizes=SIZES, repeats=REPEATS,
    k=K, seed=SEED, slow_seconds=SLOW_SECONDS,
    output_dir=OUTPUT_DIR, source_dir=ROOT / "src" / "pset2",
)
print("Submit these two files:")
print(csv_path)
print(metadata_path)


random_dna   n=    100 countNucleotides   ok (0.000007s)
random_dna   n=    100 patternIndex       ok (0.000035s)
random_dna   n=    100 patternCount       ok (0.000028s)
random_dna   n=    100 frequentWords      ok (0.000050s)
random_dna   n=    300 countNucleotides   ok (0.000006s)
random_dna   n=    300 patternIndex       ok (0.000104s)
random_dna   n=    300 patternCount       ok (0.000077s)
random_dna   n=    300 frequentWords      ok (0.000137s)
random_dna   n=   1000 countNucleotides   ok (0.000014s)
random_dna   n=   1000 patternIndex       ok (0.000532s)
random_dna   n=   1000 patternCount       ok (0.002693s)
random_dna   n=   1000 frequentWords      ok (0.001130s)
random_dna   n=   3000 countNucleotides   ok (0.000050s)
random_dna   n=   3000 patternIndex       ok (0.001028s)
random_dna   n=   3000 patternCount       ok (0.001101s)
random_dna   n=   3000 frequentWords      ok (0.001664s)
random_dna   n=  10000 countNucleotides   ok (0.000201s)
random_dna   n=  10000 patternI

In [4]:
summary = summarize(rows)
print(f"Successful measurements: {sum(r['status'] == 'ok' for r in rows)}")
print(f"Errors: {sum(r['status'] == 'error' for r in rows)}")
print(f"Skipped sizes: {sum(r['status'] == 'skipped' for r in rows)}")
for item in summary:
    print(f"{item['method']:18} {item['workload']:12} n={item['n']:>7} "
          f"median={item['median_seconds']:.6g}s  "
          f"deep={item['median_return_deep_bytes']:g} bytes  "
          f"items={item['median_return_items']:g}")


Successful measurements: 120
Errors: 0
Skipped sizes: 0
countNucleotides   random_dna   n=    100 median=7.00005e-06s  deep=496 bytes  items=4
countNucleotides   random_dna   n=    300 median=8.00006e-06s  deep=496 bytes  items=4
countNucleotides   random_dna   n=   1000 median=1.86001e-05s  deep=496 bytes  items=4
countNucleotides   random_dna   n=   3000 median=5.92002e-05s  deep=496 bytes  items=4
countNucleotides   random_dna   n=  10000 median=0.0001816s  deep=496 bytes  items=4
countNucleotides   repeated_a   n=    100 median=4.20003e-06s  deep=440 bytes  items=4
countNucleotides   repeated_a   n=    300 median=3.10014e-06s  deep=440 bytes  items=4
countNucleotides   repeated_a   n=   1000 median=7.89994e-06s  deep=440 bytes  items=4
countNucleotides   repeated_a   n=   3000 median=2.60002e-05s  deep=440 bytes  items=4
countNucleotides   repeated_a   n=  10000 median=3.82001e-05s  deep=440 bytes  items=4
frequentWords      random_dna   n=    100 median=5.19999e-05s  deep=52 bytes

In [5]:
# Optional visualization; if matplotlib is unavailable, you can install it by 
# removing the comment from the next line. Just make sure you re-comment it out afterwards!
#%pip install matplotlib
plot_results(rows, title=PARTICIPANT_ID)


ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
import shutil

# Replace with your desired zip file name (do not include .zip at the end)
output_filename = PARTICIPANT_ID + "_benchmark_results" 

# Replace with the path to the folder you want to zip
directory_to_zip = "benchmark-results" 

shutil.make_archive(output_filename, 'zip', directory_to_zip)